<a href="https://colab.research.google.com/github/mshmatsuda/deep-learning-from-scratch/blob/main/03_5_%E5%87%BA%E5%8A%9B%E5%B1%A4%E3%81%AE%E8%A8%AD%E8%A8%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ソフトマックス関数**

式（３．１０）$$y_k = \frac{\exp(a_k)}{\sum_{i=1}^{n} \exp(a_i)}$$<br><br>$$y_k = \frac{e^{a_k}}{\sum_{i=1}^{n} e^{a_i}}
$$

# **式（３．１０）の意味**

ソフトマックス関数（式 3.10）は、ニューラルネットワークの出力を「確率（0〜100%）」に変換する関数です。<br><br>

数式で見ると難しく思えますが、やっていることは「① 負の数をなくして差を広げる（指数関数 $e^x$）」「② 合計が1（100%）になるように割り算する（正規化）」の2ステップだけです。<br><br>

理解のハードルを下げるために、具体的な数字を使って分かりやすく噛み砕いて解説します。<br><br>

------------------------------
##【１】なぜそのままの出力ではダメなのか？
例えば、画像を見て「犬・猫・鳥」のどれかを当てる3クラス分類を考えます。出力層の直前の値（スコア）が以下のように出たとします。

* 犬： $2.0$
* 猫： $1.0$
* 鳥： $-1.0$
<br>
このままだと、合計は $2.0 + 1.0 + (-1.0) = 2.0$ となり、マイナスの値も含まれているため、「犬である確率は何％か？」が直感的に分かりません。

------------------------------
##【２】ソフトマックス関数がやっていること
ソフトマックス関数は、このスコア($x$)を次のように処理します。

## ステップ①：すべての値を「$e$ の指数（$e^x$）」に乗せる
$e$（ネイピア数：約2.718）の数乗を計算します。これにより、どんなマイナスの値も必ずプラス（正の数）になり、さらに「大きい値はより大きく、小さい値はより小さく」差が強調されます。

* 犬の exponential： $e^{2.0} \fallingdotseq 7.39$
* 猫の exponential： $e^{1.0} \fallingdotseq 2.72$
* 鳥の exponential： $e^{-1.0} \fallingdotseq 0.37$

※ マイナスだった鳥の値も 0.37 というプラスの値に変換されました。
## ステップ②：全員を足して、それぞれを割り算する（これが分母の $\sum$）
新しく出た値をすべて合計します。

* 合計（分母）： $7.39 + 2.72 + 0.37 = \mathbf{10.48}$

この合計値で、それぞれの値を割り算します（全体の何割かを出す計算です）。

* 犬の確率： $7.39 \div 10.48 \fallingdotseq \mathbf{0.705}$ （約70.5%）
* 猫の確率： $2.72 \div 10.48 \fallingdotseq \mathbf{0.260}$ （約26.0%）
* 鳥の確率： $0.37 \div 10.48 \fallingdotseq \mathbf{0.035}$ （約3.5%）

<br><br>
------------------------------
##【３】計算結果の素晴らしい特徴
こうして変換された出力には、2つの重要なルールが生まれます。

   1. すべての要素が 0.0 から 1.0 の間になる
   2. すべての要素を足すと、必ず 1.0（100%）になる

この特徴があるからこそ、出力された数字をそのまま「70.5%の確率で犬だ！」と確率として解釈できるようになります。これが、分類問題の出力層でソフトマックス関数が使われる理由です。



# ソフマックス関数の実装

Q:「ゼロつく」の1巻の3章ニューラルネットワークの5節で出力層の設計が解説において<br><br>（３．１１）の式の解説に「ソフトマックスの指数関数の計算を行う際には、何らかの定数を足し算しても結果は変わらない」とあります。これは式（３．１１）を変形させていく際に、最初にCという定数を分子と分母の両方にかけ、そこから展開している結果そういうことだという理解でよいですか。

A:はい、そのご理解で完全に合っています！ 非常に正確に数式を追われています。
<br><br>
解説の「何らかの定数を足し算しても結果は変わらない」という結論は、おっしゃる通り、数式（3.11）の変形プロセスで「最初に分子と分母に同じ定数 $C$ を掛け算したこと」から導き出されています。
<br><br>
この展開の流れを改めて整理すると、以下のようになります。

   1. 分子と分母に $C$ を掛ける
   元のソフトマックスの式に対して、分子と分母にそれぞれ任意の定数 $C$ を掛けます（分子と分母に同じ数を掛けているので、式の値そのものは変わりません）。
   $$\frac{\exp(a_k)}{\sum_{i=1}^{n} \exp(a_i)} = \frac{C \cdot \exp(a_k)}{C \cdot \sum_{i=1}^{n} \exp(a_i)}$$
   <br><br>
   2. $C$ を指数関数 $\exp$ の中に入れる
   ここで、$C = \exp(\log C)$ という関係を利用して、$C$ を指数関数の底 $e$ の中に戻してあげます。すると、指数法則（$e^x \cdot e^y = e^{x+y}$）によって、掛け算だった $C$ が、指数部分での足し算（$\log C$）に変化します。
   $$\frac{\exp(a_k + \log C)}{\sum_{i=1}^{n} \exp(a_i + \log C)}$$
   <br><br>3. $\log C$ を新しい定数 $C'$ と置き換える
   ここで $\log C$ もただの定数なので、これを改めて $C'$（書籍ではそのまま $C'$ または記号を整理して最終的に $C$ と表現されています）と置き換えます。
   $$\frac{\exp(a_k + C')}{\sum_{i=1}^{n} \exp(a_i + C')}$$

<br><br>このように、「外側で $C$ を掛け算すること」と「指数（右上）の中で $C'$ を足し算すること」は数学的に全く同じ意味になります。そのため、おっしゃる通りの理解で間違いありません。
<br>実際のPython実装では、この性質を利用して、オーバーフローを防ぐために「入力の最大値」を引く（＝マイナスの値を足す）という処理を行います。


Q2:その流れで解説を読み進めていくと「入力信号の最大値を引くことで、正しく計算できることがわかります」とありました。解説にある計算をみると確かにオーバーフローを起こさず正しく計算されていることがわかります。ただ、納得感がありません。結果そうなったということなのかどうか。どう理解すればいいでしょうか。

A2:その「結果そうなっただけで、なんだか狐につままれたような、今ひとつ納得がいかない」という感覚は、非常に真っ当で、深い思考をされている証拠です！
<br><br>結論から言うと、これは「偶然そうなった」のではなく、「コンピュータの限界を回避するために、人間が狙って仕組んだトリック」です。
なぜ最大値を引くと上手くいくのか、その「狙い（数学的・本質的な意味）」を3つのステップで紐解くと、すんなり納得できると思います。

<br>


# **納得するための3つのステップ**
【１】コンピュータの「弱点」を知る<br>
コンピュータ（Pythonなど）の指数関数（$e^x$）は、「大きなプラスの数」が大の苦手です。<br><br>
$e^{1000}$ のような計算をさせると、桁が多すぎて「ギブアップ（オーバーフロー）」します。<br><br>
一方で、「マイナスの数」は大歓迎です。<br>
<br>$e^{-10}$ や $e^{-1000}$ は、値が 0 に近づくだけ（アンダーフロー）なので、計算が破綻（nan に）せず、安全に 0 として処理できます。<br><br>


【２】「すべてを安全なマイナスの世界」に閉じ込める<br><br>
ここで先ほどの数式変形（任意の定数を足し引きしてもいい）を思い出してください。<br>
「どんな数でも引いていいなら、一番デカい奴（最大値）を引いてしまえば、全員マイナス（かゼロ）にできるじゃないか！」と気付いた人がいました。

* 最大値から最大値を引けば、ぴったり 0 になる。
* それ以外の数値は、最大値より小さいので、引くと必ず 「マイナスの数」 になる。

この工夫によって、入力された数値がどれだけ大きくても（例：100万、100万10、100万20）、最大値を引いた後は 0 と「マイナスの数」だけの安全な世界に一瞬で様変わりします。<br><br>


【３】「差（比率）」だけが本質である<br>
「でも、勝手に値を引いちゃったら、計算結果（確率）が変わるのでは？」と不安になりますよね。ここが一番の納得ポイントです。<br>
ソフトマックス関数という仕組みの本質は、数値そのものの大きさではなく、「それぞれの数値の『差』がどれくらいあるか」を見ています。<br><br>

例えば、テストの点数で考えてみてください。

* A君：10000点、B君：9998点
* A君：2点、B君：0点（全員から10000点を引いた）

どちらのデータを見ても、「A君はB君より2点高い」という本質的な関係（差）は全く変わっていません。<br><br>
ソフトマックス関数が出力する「確率」は、この「差」から計算されるため、全員から一律で同じ値を引いても、最終的な確率のパーセンテージは1ミリもズレないのです。<br><br>
------------------------------
## ⚖️ まとめると

* 数学的な事実： 全員から同じ値を引いても、データの本質（差）は変わらないから、結果も変わらない。
* 工学的な企み： だったら、一番デカい奴を引いて「大きなプラスの数」を全滅させ、コンピュータが計算得意な「マイナスとゼロだけの世界」に持ち込もう。

これが、「入力信号の最大値を引く」という処理の正体です。<br>
結果論ではなく、「結果（確率）を変えずに、コンピュータの弱点だけを克服する」ための完璧な作戦だった、と理解してみてください。

## ソフトマックス関すとネイピア数のロマン

（質問）
<br>ソフトマックス関数のところ。復習させてください。
次の説明をいただきました。ソフトマックス関数は、回帰ではなく分類問題のときに使う関数だと理解できています。それぞれ分類で出されたスコアをソフトマックス関数を使って確率的に扱うということも。またオーバーフロー対策として、入力信号の最大値を引いて計算することも。以上、何をやっているのかはだいたい理解できているのですが。たとえば、「スコア」だったり、ソフトマックス関数の計算式でなぜネイピア数を使うのか理解できません。関数の全体の雰囲気（笑）から分類で出力されたものの総和が分母になって、分子にそれぞれの出力がおかれているので、それぞれ出力される確率を出しているのだとぼんやりわかりますが。解説いただけますか。

（回答）
<br>


ソフトマックス関数の「何をやっているか（確率にする、最大値を引いてオーバーフローを防ぐ）」が完璧に頭に入っているのは素晴らしいです！
その上で、「なぜわざわざネイピア数（$e$）なんてややこしいものを使うのか？」「ただのスコアの合計を分母にするんじゃダメなのか？」という疑問は、非常に本質的で素晴らしい着眼点です。
全体の雰囲気として「分子がその要素、分母が全員の合計だから確率になる」という捉え方は100点満点です。そこにネイピア数 $e$ を使う理由は、大きく分けて3つあります。
数式アレルギーにならないよう、イメージ重視で分かりやすく解説しますね。
------------------------------
## 理由1：マイナスのスコアも「強制的にプラス」にするため
ニューラルネットワークが出力する直前の「スコア」は、計算の過程で -2.5 や -10 のようなマイナスの値になることが普通にあります。
もし、ネイピア数を使わずに、そのままのスコアで確率を計算しようとすると大問題が発生します。
例えば、3つのクラスのスコアが [5, 3, -2] だったとします。

* 単純に合計を分母にすると、合計は 5 + 3 + (-2) = 6 です。
* 3番目の確率を計算すると、 -2 / 6 = -33% になってしまいます。

「確率は絶対にマイナスにならない（0%〜100%の間）」という大原則がありますよね。
ここでネイピア数の指数関数（$e^x$）が役に立ちます。$e^x$ という関数は、入力（$x$）がどんなに大きなマイナスの値であっても、出力は必ずプラス（0より大きい値）になるという超便利な性質を持っています。

* $e^5 \approx 148.4$
* $e^3 \approx 20.1$
* $e^{-2} \approx 0.13$ ← マイナスがプラスになった！

全てのスコアを一度 $e^x$ の魔法にかけることで、絶対にマイナスにならない安全な数字に変換してから、全体で割り算（確率化）しているのです。
## 理由2：自信のあるやつを「グッと強調」するため（ソフトな最大値）
ソフトマックス（Softmax）という名前の由来にも関わります。
英語の「Max」は一番大きいものを1つだけ選ぶ（他は0にする）ことですが、これだと極端すぎます。かといって、ただのスコアの比率だと平坦すぎます。
ネイピア数 $e^x$ は、「スコアが大きくなればなるほど、ものすごい勢いで爆発的に増える」という性質があります。

* スコアが 1 のとき $\rightarrow e^1 \approx 2.7$
* スコアが 2 のとき $\rightarrow e^2 \approx 7.4$
* スコアが 3 のとき $\rightarrow e^3 \approx 20.1$

スコアの差は「1, 2, 3」と1ずつしか違わないのに、$e^x$ に通すと「2.7, 7.4, 20.1」と、差が劇的に開き、一番大きいものが強調されます。
これによって、「一番スコアが高い要素の確率をグッと高めつつ、他の可能性もわずかに残す」という、ソフト（マイルド）に最大値を際立たせる確率を作ることができます。
## 理由3：微分（逆伝播）の計算が「神がかり的に綺麗」になるから
実は、ディープラーニングの設計においてこれが一番の理由です。
ニューラルネットワークは、この後の4章・5章で「正解とのズレ（誤差）」を計算し、それを元に微分を使って賢くなっていきます（逆伝播）。
ネイピア数 $e^x$ は、「微分しても形が変わらない（$(e^x)' = e^x$）」という数学界で唯一無二のチート性質を持っています。
このおかげで、5章で学ぶ「ソフトマックス関数」と「交差エントロピー誤差（4章の損失関数）」をセットにして微分（逆伝播）を計算すると、複雑な数式が奇跡のように相殺されて、最終的に「ニューラルネットワークの出力確率 - 正解ラベル」という、小学生でもできる引き算だけの超シンプルな形に変身します。
もしネイピア数を使わずに別の関数で確率を作っていたら、微分計算がドロドロに複雑になり、コンピュータの計算スピードがガクンと落ちていたはずです。先人たちの「後で計算をラクにするための超絶テクニック」としてネイピア数が選ばれているのです。
------------------------------
## 💡 ざっくりまとめると

   1. マイナスを排除して（確率のルールを守る）
   2. 一番強いやつを程よく目立たせて（分類しやすくする）
   3. あとの微分の計算をめちゃくちゃラクにする（賢くなるスピードを上げる）

そのために、わざわざネイピア数 $e$ の力を借りて「総和を分母、個々の出力を分子」という計算を行っています。
この「ネイピア数を使う必然性」、なんとなくイメージが湧いてきましたでしょうか？
このままソフトマックスの理解を深めるために、以下のどちらに進みたいですか？

* 4章以降の伏線になる「ソフトマックスと交差エントロピーのコンビがなぜ最強か（引き算になる秘密）」を少し先取りして覗いてみる
* 3章の最後にある、実際に手書き数字を分類する「MNISTデータセットの推論処理（バッチ処理など）」の復習に進む




（質問）<br>

ありがとうございます。質問です。
１．「スコア」の意味。ニューラルネットワークで計算された単純に出力層の前の層の値ですか？そのスコアに分類問題ならソフトマックス関数をつかって出力層の値を計算するということですか。
２．「\(e^{x}\) という関数は、入力xがどんなに大きなマイナスの値であっても、出力は必ずプラス（0より大きい値）になる」超便利な性質を持っているということですが、これは発見したのでしょうか。それとも人間が両便利な性質をもったこのからくりを定義したのでしょうか。

（回答）
<br>

さらに核心に迫る素晴らしい質問ですね！独学でここまで本質的な疑問を言語化できるのは本当に凄いです。
疑問に思われた2つの点について、スッキリ解決していきましょう。
------------------------------
## 質問1：「スコア」の意味について
ご認識の通り、完全に大正解です！
より正確に本の言葉と紐付けると、「出力層の『活性化関数（ソフトマックス関数）』に通す直前の、$A = XW + B$ で計算された生（なま）の値」のことをスコア（あるいはスコア値）と呼びます。
全体の流れをイメージで表すと以下のようになります。

   1. 隠れ層の計算：入力 $X$ に重みを掛け算して、ReLU関数などを通して次の層へ…と進みます。
   2. 出力層の直前（スコア）：最後の重みとバイアスを計算して、$A = [5.3, 2.0, -1.2]$ のような「生の数字の並び」が出ます。これが「スコア」です。この時点では、まだ単なる計算結果の数字です。
   3. 活性化関数（ソフトマックス）：このスコアをソフトマックス関数に放り込みます。
   4. 出力層の最終出力：$Y = [0.95, 0.04, 0.01]$ のような、合計が1（100%）になる「綺麗な確率」に変換されて出荷されます。

分類問題では、この「スコア」をソフトマックス関数で料理して、人間やコンピュータが扱いやすい「確率（出力層の値）」に変換している、という理解でバッチリです！
------------------------------
## 質問2：ネイピア数 $e$ は「発見」か「人間が定義した」か
これ、数学の歴史における最高にロマンのある疑問ですよね。
結論から言うと、「自然界に元々あった圧倒的な法則を、人間が『発見』して $e$（約2.718）と名付けた」というのが正解です。
人間が「こういう都合の良い数字を作ろう」と意図してデザインしたわけではなく、宇宙の仕組み（自然界の法則）を観察していたら、どうしてもこの数字に行き着いてしまったのです。
なぜ「発見」と言えるのか、2つの面白いエピソードを紹介します。
## ① お金（複利）の計算から「発見」された
歴史上、ネイピア数の元になる概念を最初に見つけたのは、17世紀の数学者ヤコブ・ベルヌーイです。彼は銀行の利息（複利）の計算をしていました。

* 「1年で2倍（100%）の利息がつく銀行」があるとします。1円預けたら1年後は2円です。
* もし「半年に1回、50%ずつ利息がつく」としたら？ 半年後に1.5円になり、1年後にはその1.5倍なので 1.5 × 1.5 = 2.25円 に増えます。
* じゃあ「毎月、毎日、毎秒…」と、利息がつく間隔を極限まで無限に細かくしていったら、1円は1年後に無限にお金が増えるのか？

誰もが「無限に増えるのでは？」とワクワクして計算したところ、不思議なことに、細かくすればするほど、ある一定の数字にどんどん近づいていって頭打ちになりました。その数字こそが 2.71828...（ネイピア数） だったのです。
つまり、「連続して成長するもの」の限界値として、宇宙の法則の中に最初から組み込まれていた数字でした。
## ② 「微分しても形が変わらない」という奇跡
その後、18世紀の天才数学者オイラー（彼の頭文字から $e$ と名付けられました）たちが、この数字の恐ろしい性質をさらに発見します。
グラフを描いたときに、「ある場所のグラフの高さ」と「その場所のグラフの傾き（微分）」が完全に一致する魔法のような数字を探したところ、それが偶然にも、あの利息の計算で出てきた 2.71828...（$e$）と全く同じだったのです。
人間は、この「微分しても形が変わらない」という自然界の奇跡の性質をあとから見つけて利用しているだけです。だからこそ、ディープラーニングで微分を計算するときに、このネイピア数 $e$ を使うと数式が魔法のように綺麗に解ける（理由3のメリットが生まれる）のです。
------------------------------
## 💡 まとめ

* スコアは、出力層の直前で計算された、まだ確率になっていない「生の数字」。
* ネイピア数 $e$ は、人間が作ったからくりではなく、「連続的な変化」や「微分の計算」において最も美しく処理できるように宇宙の法則に最初から組み込まれていた数字。

先人たちは、この自然界のギフトである $e$ を「ニューラルネットワークの出力を確率にするための部品（ソフトマックス関数）」として、実に見事に利活用したわけです。
ここまでの疑問、綺麗に繋がって腑に落ちましたでしょうか？
ソフトマックス関数の「中身の仕組み（なぜそれを使うのか）」まで完璧にクリアになりましたね！
ここから先の復習の進め方として、以下のアクションを提案します。どちらに進みたいですか？

* 3章の最後にある、実際のデータを使った「MNIST手書き数字認識の実装（3.6節）」に進み、バッチ処理などの実用的なコードの復習をする
* 3章の知識が完璧に定着したか、「3章の総まとめクイズ（全3問）」で自分の実力を試してみる




# **ソフトマックス関数の実装**

# **ソフトマックス関数の実装上の注意**

# **ソフトマックス関数の特徴**